In [214]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [215]:
df = pd.read_csv("../Data/master.csv")

In [216]:
df.columns

Index(['scraped_at', 'source_url', 'page', 'url', 'make_model', 'trim',
       'price', 'mileage', 'transmission', 'fuel_type', 'Dealer Name',
       'Dealer Address', 'Dealer URL'],
      dtype='object')

In [217]:
#How Many Duplicate Cars we have:
duplicate_count=df["url"].duplicated().sum()
print(f"Number of duplicate URLs: {duplicate_count}")
df = df.drop_duplicates(subset=["url"],keep="last")

Number of duplicate URLs: 252


<p>There are so many values in price column as Price on Request, about 13% in the initial dataset, I dropped them all.</p>

In [218]:
print(f"In this DataSet We have this many NAN {np.round(df["price"].isna().sum() / df["price"].shape[0]*100)} %")

In this DataSet We have this many NAN 4.0 %


In [219]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 55042 entries, 0 to 55293
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   scraped_at      55041 non-null  object 
 1   source_url      55041 non-null  object 
 2   page            55041 non-null  float64
 3   url             55041 non-null  object 
 4   make_model      55041 non-null  object 
 5   trim            51116 non-null  object 
 6   price           53103 non-null  object 
 7   mileage         52539 non-null  object 
 8   transmission    52286 non-null  object 
 9   fuel_type       52583 non-null  object 
 10  Dealer Name     20055 non-null  object 
 11  Dealer Address  20055 non-null  object 
 12  Dealer URL      7112 non-null   object 
dtypes: float64(1), object(12)
memory usage: 5.9+ MB


In [220]:
Year = pd.DataFrame(columns=["Year","Make","Model"])
df[["Year","Make","Model"]]=df["make_model"].str.split(" ",expand=True,n=2)
df.head()


,scraped_at,source_url,page,url,make_model,trim,price,mileage,transmission,fuel_type,Dealer Name,Dealer Address,Dealer URL,Year,Make,Model
0,2026-05-20 16:34,https://www.autotrader.ca/cars/on/toronto/,1.0,https://www.autotrader.ca/offers/lincoln-corsa...,2022 Lincoln Corsair,NaN,"$27,999","109,941 km",Automatic,Gas,NaN,NaN,NaN,2022,Lincoln,Corsair
1,2026-05-20 16:34,https://www.autotrader.ca/cars/on/toronto/,1.0,https://www.autotrader.ca/offers/jeep-cherokee...,2026 Jeep Cherokee,Laredo,"$47,660",50 km,Automatic,Gas/Electric Hybrid,NaN,NaN,NaN,2026,Jeep,Cherokee
2,2026-05-20 16:34,https://www.autotrader.ca/cars/on/toronto/,1.0,https://www.autotrader.ca/offers/jeep-cherokee...,2026 Jeep Cherokee,LAREDO BlindSpot and CrossPath Detection Drowsy,"$47,198",18 km,Automatic,Gas/Electric Hybrid,NaN,NaN,NaN,2026,Jeep,Cherokee
4,2026-05-20 16:34,https://www.autotrader.ca/cars/on/toronto/,1.0,https://www.autotrader.ca/offers/hyundai-tucso...,2026 Hyundai Tucson,Preferred AWD w/Trend Pkg,"$42,219",150 km,Automatic,Gas,NaN,NaN,NaN,2026,Hyundai,Tucson
5,2026-05-20 16:34,https://www.autotrader.ca/cars/on/toronto/,1.0,https://www.autotrader.ca/offers/mazda-3-sport...,2021 Mazda Mazda3 Sport,SPORT GT | Nav | Leather | Sunroof | BSM | Car...,NaN,"129,000 km",Automatic,Gas,NaN,NaN,NaN,2021,Mazda,Mazda3 Sport


In [221]:
#Price needs to get converted from object to float.
df["price"]=df["price"].str.replace('$',"")
df["price"]=df["price"].str.replace(',',"")
df["price"]=pd.to_numeric(df["price"],errors="coerce")
df.dropna(subset=["price"],inplace=True)
print(f"In this DataSet We have {np.round(df["price"].isna().sum() / df["price"].shape[0]*100)} %")

In this DataSet We have 0.0 %


In [222]:
#I need to convert mileage to float.
df["mileage"]=df["mileage"].str.replace("km","")
df["mileage"]=df["mileage"].str.replace(",","")
condition =df["mileage"].isna() & df["Year"].isin(["2026","2027"])
df.loc[condition,"mileage"] = 0
df.dropna(subset=["mileage"], inplace=True)
df["mileage"]=df["mileage"].astype(float)
df["Year"] = pd.to_numeric(df["Year"],errors="coerce")

In [223]:
#I need to Make condition Columns to have a New and old cars seperated.
condition  = df["Year"].isin([2026 & 2027]) & (df["mileage"]<=200)
df.loc[condition, "Condition"] = "New"
df.loc[~condition, "Condition"] = "Used"
df.head()

,scraped_at,source_url,page,url,make_model,trim,price,mileage,transmission,fuel_type,Dealer Name,Dealer Address,Dealer URL,Year,Make,Model,Condition
0,2026-05-20 16:34,https://www.autotrader.ca/cars/on/toronto/,1.0,https://www.autotrader.ca/offers/lincoln-corsa...,2022 Lincoln Corsair,NaN,27999.0,109941.0,Automatic,Gas,NaN,NaN,NaN,2022.0,Lincoln,Corsair,Used
1,2026-05-20 16:34,https://www.autotrader.ca/cars/on/toronto/,1.0,https://www.autotrader.ca/offers/jeep-cherokee...,2026 Jeep Cherokee,Laredo,47660.0,50.0,Automatic,Gas/Electric Hybrid,NaN,NaN,NaN,2026.0,Jeep,Cherokee,New
2,2026-05-20 16:34,https://www.autotrader.ca/cars/on/toronto/,1.0,https://www.autotrader.ca/offers/jeep-cherokee...,2026 Jeep Cherokee,LAREDO BlindSpot and CrossPath Detection Drowsy,47198.0,18.0,Automatic,Gas/Electric Hybrid,NaN,NaN,NaN,2026.0,Jeep,Cherokee,New
4,2026-05-20 16:34,https://www.autotrader.ca/cars/on/toronto/,1.0,https://www.autotrader.ca/offers/hyundai-tucso...,2026 Hyundai Tucson,Preferred AWD w/Trend Pkg,42219.0,150.0,Automatic,Gas,NaN,NaN,NaN,2026.0,Hyundai,Tucson,New
6,2026-05-20 16:34,https://www.autotrader.ca/cars/on/toronto/,1.0,https://www.autotrader.ca/offers/mazda-cx-5-su...,2024 Mazda CX-5,Suna AWD/CLEAN CARFAX/LOW MILEAGE/SERVICE RECORD,36998.0,38480.0,Automatic,Gas,NaN,NaN,NaN,2024.0,Mazda,CX-5,Used


In [224]:
df[df["Condition"] == "Used"].shape[0]

29617

In [225]:
# --- Mileage ---
mileage_bins = pd.cut(df["mileage"], bins=range(0, int(df["mileage"].max()) + 100000, 100000))
mileage_counts = mileage_bins.value_counts()

valid_mileage_bins = mileage_counts[mileage_counts >= 100].index
df = df[mileage_bins.isin(valid_mileage_bins)]

price_bins = pd.cut(df["price"], bins=range(0, int(df["price"].max()) + 5000, 5000))
price_counts = price_bins.value_counts()

valid_price_bins = price_counts[price_counts >= 100].index
df = df[price_bins.isin(valid_price_bins)]

In [226]:
df

,scraped_at,source_url,page,url,make_model,trim,price,mileage,transmission,fuel_type,Dealer Name,Dealer Address,Dealer URL,Year,Make,Model,Condition
0,2026-05-20 16:34,https://www.autotrader.ca/cars/on/toronto/,1.0,https://www.autotrader.ca/offers/lincoln-corsa...,2022 Lincoln Corsair,NaN,27999.0,109941.0,Automatic,Gas,NaN,NaN,NaN,2022.0,Lincoln,Corsair,Used
1,2026-05-20 16:34,https://www.autotrader.ca/cars/on/toronto/,1.0,https://www.autotrader.ca/offers/jeep-cherokee...,2026 Jeep Cherokee,Laredo,47660.0,50.0,Automatic,Gas/Electric Hybrid,NaN,NaN,NaN,2026.0,Jeep,Cherokee,New
2,2026-05-20 16:34,https://www.autotrader.ca/cars/on/toronto/,1.0,https://www.autotrader.ca/offers/jeep-cherokee...,2026 Jeep Cherokee,LAREDO BlindSpot and CrossPath Detection Drowsy,47198.0,18.0,Automatic,Gas/Electric Hybrid,NaN,NaN,NaN,2026.0,Jeep,Cherokee,New
4,2026-05-20 16:34,https://www.autotrader.ca/cars/on/toronto/,1.0,https://www.autotrader.ca/offers/hyundai-tucso...,2026 Hyundai Tucson,Preferred AWD w/Trend Pkg,42219.0,150.0,Automatic,Gas,NaN,NaN,NaN,2026.0,Hyundai,Tucson,New
6,2026-05-20 16:34,https://www.autotrader.ca/cars/on/toronto/,1.0,https://www.autotrader.ca/offers/mazda-cx-5-su...,2024 Mazda CX-5,Suna AWD/CLEAN CARFAX/LOW MILEAGE/SERVICE RECORD,36998.0,38480.0,Automatic,Gas,NaN,NaN,NaN,2024.0,Mazda,CX-5,Used
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55288,2026-07-09 17:15:02,https://www.autotrader.ca/cars/on/toronto/,6.0,https://www.autotrader.ca/offers/subaru-forest...,2018 Subaru Forester,2.0XT Touring CVT,18995.0,131300.0,Automatic,Gas,XTREME MOTORS LTD,"KITCHENER, ON • 92 km from you",NaN,2018.0,Subaru,Forester,Used
55289,2026-07-09 17:15:02,https://www.autotrader.ca/cars/on/toronto/,6.0,https://www.autotrader.ca/offers/nissan-sentra...,2012 Nissan Sentra,NaN,5800.0,172000.0,Automatic,Gas,BEST DEAL AUTO INC,"Hamilton, ON • 63 km from you",NaN,2012.0,Nissan,Sentra,Used
55290,2026-07-09 17:15:02,https://www.autotrader.ca/cars/on/toronto/,6.0,https://www.autotrader.ca/offers/lexus-rx-prem...,2022 Lexus RX,"Premium AWD, Back Up Cam, Sunroof, Bluetooth",42995.0,64028.0,Automatic,Gas,Auto Planet Direct,"Brampton, ON • 24 km from you",https://www.autotrader.ca/dealers/47940850,2022.0,Lexus,RX,Used
55291,2026-07-09 17:15:02,https://www.autotrader.ca/cars/on/toronto/,6.0,https://www.autotrader.ca/offers/audi-a5-techn...,2018 Audi A5 Cabriolet,Technik,26995.0,198000.0,Automatic,Gas,NaN,NaN,NaN,2018.0,Audi,A5 Cabriolet,Used


In [227]:
df.to_csv("PreProcessVersion.csv")